In [1]:
import pandas as pd
import numpy as np
import joblib

In [2]:
xgb = joblib.load('C:\\Users\\NOTHING\\Pictures\\ValueLens AI\\models\\clv_model.pkl')
df=pd.read_csv('C:\\Users\\NOTHING\\Pictures\\ValueLens AI\\data\\processed\\customer_features.csv')

feature_cols = ['Recency', 'Frequency', 'Monetary', 'Tenure', 'AvgOrderValue', 'ProductDiversity', 'WeekendRatio']
X = df[feature_cols]

In [3]:
df['predicted_clv'] = np.expm1(xgb.predict(X))

In [4]:
df

,Customer ID,Recency,Frequency,Monetary,Tenure,AvgOrderValue,ProductDiversity,WeekendRatio,future_6m_clv,predicted_clv
0,12346,134,8,77556.46,534,9694.5575,27,0.000000,0.00,2.969319
1,12347,55,4,2434.96,213,608.7400,90,0.322581,2486.57,886.202759
2,12348,57,4,1709.40,247,427.3500,25,0.000000,310.00,160.948425
3,12349,216,3,2671.14,398,890.3800,90,0.000000,1757.55,561.174255
4,12350,119,1,334.40,119,334.4000,17,0.000000,0.00,8.104921
...,...,...,...,...,...,...,...,...,...,...
4930,18283,9,10,1134.89,467,113.4890,240,0.225513,1530.01,736.308105
4931,18284,240,1,461.68,240,461.6800,28,0.000000,0.00,6.340496
4932,18285,469,1,427.00,469,427.0000,12,0.000000,0.00,1.518838
4933,18286,285,2,1296.43,532,648.2150,67,0.000000,0.00,5.382359


In [5]:
print(df['predicted_clv'].describe())
print(df['predicted_clv'].quantile([0.5, 0.7, 0.9]))

count     4935.000000
mean       522.062012
std       3254.785645
min         -0.715460
25%          3.396019
50%         19.844057
75%        195.714813
max      87554.796875
Name: predicted_clv, dtype: float64
0.5     19.844057
0.7    117.517976
0.9    949.567908
Name: predicted_clv, dtype: float64


In [6]:
vip_threshold = df['predicted_clv'].quantile(0.90)
loyal_threshold = df['predicted_clv'].quantile(0.65)
growth_threshold = df['predicted_clv'].quantile(0.30)

print(vip_threshold, loyal_threshold, growth_threshold)

949.567907714844 67.98388977050782 4.814062404632569


In [7]:
def assign_segment(clv, vip_threshold, loyal_threshold, growth_threshold):
    if clv >= vip_threshold:
        return 'VIP'
    elif clv >= loyal_threshold:
        return 'Loyal'
    elif clv >= growth_threshold:
        return 'Growth'
    else:
        return 'At Risk'

df['segment'] = df['predicted_clv'].apply(
    lambda clv: assign_segment(clv, vip_threshold, loyal_threshold, growth_threshold)
)

In [9]:
df['predicted_clv'] = df['predicted_clv'].clip(lower=0)

In [10]:
print(df['segment'].value_counts())
print(df.groupby('segment')['predicted_clv'].describe())

segment
Growth     1727
At Risk    1481
Loyal      1233
VIP         494
Name: count, dtype: int64
          count         mean          std         min          25%  \
segment                                                              
At Risk  1481.0     1.787201     1.358312    0.000000     0.651092   
Growth   1727.0    21.869024    16.051754    4.818782     8.913313   
Loyal    1233.0   336.695129   242.730957   67.990166   132.246964   
VIP       494.0  4293.158203  9478.847656  949.790344  1289.597015   

                 50%          75%           max  
segment                                          
At Risk     1.407542     2.793935      4.812882  
Growth     16.570480    30.722653     67.983192  
Loyal     261.405487   486.594971    949.234253  
VIP      1860.723816  2911.922607  87554.796875  


In [11]:
# checking

from valuelens.segmentation import add_segments
from valuelens.recommender import add_recommendations

df, thresholds = add_segments(df)
df = add_recommendations(df)

print(df[['Customer ID', 'predicted_clv', 'segment', 'recommended_action']].head(10))
print(df['recommended_action'].value_counts())

   Customer ID  predicted_clv  segment       recommended_action
0        12346       2.969319  At Risk  Win-back email campaign
1        12347     886.202759    Loyal          Loyalty rewards
2        12348     160.948425    Loyal          Loyalty rewards
3        12349     561.174255    Loyal          Loyalty rewards
4        12350       8.104921   Growth   Personalized discounts
5        12351       6.715837   Growth   Personalized discounts
6        12352     420.925140    Loyal          Loyalty rewards
7        12353       3.806962  At Risk  Win-back email campaign
8        12354      91.670425    Loyal          Loyalty rewards
9        12355      11.422796   Growth   Personalized discounts
recommended_action
Personalized discounts            1727
Win-back email campaign           1481
Loyalty rewards                   1233
Early access + premium support     494
Name: count, dtype: int64


In [12]:
from valuelens.roi_simulator import simulate_roi

everyone = simulate_roi(df, target_segments=['VIP', 'Loyal', 'Growth', 'At Risk'])
focused = simulate_roi(df, target_segments=['VIP', 'Loyal'])

print("Target everyone:", everyone)
print("Target VIP + Loyal only:", focused)

Target everyone: {'targeted_customers': 4935, 'baseline_revenue': np.float32(2.57638e+06), 'incremental_revenue': np.float32(144181.8), 'campaign_cost': 42201.0, 'net_profit': np.float32(101980.8), 'roi_percent': np.float32(241.65)}
Target VIP + Loyal only: {'targeted_customers': 1727, 'baseline_revenue': np.float32(2.5359652e+06), 'incremental_revenue': np.float32(139252.62), 'campaign_cost': 13575.0, 'net_profit': np.float32(125677.62), 'roi_percent': np.float32(925.8)}
